Sustainability Recommendation System

The last stage of this project is to turn the information from the previous notebooks into actionable recommendations.

These machine learning models idenfity important predictors of carbon footprint within the dataset, but simply identifying these features does not reduce the problem. This notebook takes the imporant predictors from before and recommends unique action based on the users behavioral information that may reduce environmental impact.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

In [2]:
df = pd.read_csv('../data/personal_carbon_footprint_behavior.csv')

In [16]:
model = LinearRegression()

df = pd.read_csv('personal_carbon_footprint_behavior.csv')

df_dummies = df.copy()
df_dummies = df_dummies.drop(columns=('user_id'))
df_dummies = pd.get_dummies(df_dummies, dtype=int)

model = model.fit(df_dummies[["distance_km", "electricity_kwh", "renewable_usage_pct", "screen_time_hours",
                              "waste_generated_kg", "eco_actions", "day_type_Weekday", "day_type_Weekend",
                              "transport_mode_Bike", "transport_mode_Bus", "transport_mode_Car", "transport_mode_EV",
                              "transport_mode_Walk", "food_type_Mixed", "food_type_Non-Veg", "food_type_Veg",
                              "carbon_impact_level_High", "carbon_impact_level_Low", "carbon_impact_level_Medium"]],df_dummies[["carbon_footprint_kg"]])


sample_data = pd.DataFrame(df_dummies.drop(columns=('carbon_footprint_kg')))
sample_data["Footprint Prediction"] = model.predict(sample_data)

X = df_dummies.drop('carbon_footprint_kg', axis=1)
y = df_dummies['carbon_footprint_kg']

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [17]:
behaviors = df[['distance_km','electricity_kwh','renewable_usage_pct','screen_time_hours','waste_generated_kg','eco_actions']].copy()
scaler = StandardScaler()
scaled_behaviors = scaler.fit_transform(behaviors)
kmeans = KMeans(init='random', n_clusters=3, n_init=10,random_state=42)
behaviors['cluster'] = kmeans.fit_predict(scaled_behaviors)

cluster_group = behaviors.groupby('cluster').mean()
cluster_group_r = cluster_group.rename(index={0:'Lesser Consumers (Cluster 0)', 1 :'Large Consumers (Cluster 1)', 2 : 'Renewable Consumers (Cluster 2)'})

df["cluster"] = behaviors["cluster"]

Recommendations

Each of these recommendations are based on cluster (above) or individual user values. The recommendations below are applied to clusters and individual user data rather than using a complete recommendation algorithm becausr the information derived from previous notebook analysis.

In [18]:
recommendation = {
    0:["Increase biking or walking distances", "Increase daily eco-friendly actions", "Reduce screen time to save electricity"],
    1:["Increase public transportation usage or start walking and biking", "Increase renewable energy usage", "Be conscientious about waste generated"],
    2:["Increase biking or walking distances","Be conscientious about waste generated"]
}
cluster_names = {
    0: "Lesser Consumers",
    1: "Large Consumers",
    2: "Renewable Consumers"
}

def recommend(cluster):
    return recommendation[cluster]

def recommend_user(user):
    recs = []

    if user["distance_km"] > 9:
        recs.append("Use public transportation multiple times per week.")

    if user["electricity_kwh"] > 6:
        recs.append("Reduce unnecessary appliance usage.")

    if user["waste_generated_kg"] > 5:
        recs.append("Increase recycling and composting.")

    if user["eco_actions"] < 0.5:
        recs.append("Try completing one additional eco-friendly action each day.")

    return recs



In [19]:
    def future_user_prediction(user):

        future_user = user.copy()

        if future_user["distance_km"] > 9:
            future_user["distance_km"] *= 0.85

        if future_user["electricity_kwh"] > 4.5:
            future_user["electricity_kwh"] *= 0.9

        if future_user["renewable_usage_pct"] < 25:
            future_user["renewable_usage_pct"] == 25

        if future_user["waste_generated_kg"] > 0.6:
            future_user["waste_generated_kg"] *= 0.75

        if future_user["eco_actions"] < 0.5:
            future_user["eco_actions"] += 1

        return future_user
    

    def setup_predict(user):
        feature_columns = X_train.columns

        user_df = pd.DataFrame([user])

        user_df = user_df.drop(columns=["user_id", "cluster"], errors="ignore")

        user_df = pd.get_dummies(user_df, dtype=int)

        user_df = user_df.reindex(columns=feature_columns, fill_value=0)

        return user_df

The future user scenario takes into account the new carbon footprint predictoin if a user follows the individual behavioral changes the model suggests. These new values are then fed into the regression to estimate the possible footprint reduction. This does, however, assume that users will follow the recommendations and keep current relationships stable. 

In [ ]:

sample = df.sample(3, random_state=42)

for index, user in sample.iterrows():

    present_user = user.copy()
    future_user = future_user_prediction(user)

    current_prediction = model.predict(setup_predict(present_user))[0]
    future_prediction = model.predict(setup_predict(future_user))[0]

    reduction = current_prediction - future_prediction
    percent = abs((reduction / current_prediction)*100)

    print("="*60)

    print(f"User {index}")
    print(f"Sustainability Profile: {cluster_names[user['cluster']]}")
    #print(f"Current Carbon Footprint: {user['carbon_footprint_kg']:.2f} kg CO\u2082")
    print(f"Current Prediction Carbon Footprint: \033[33m{current_prediction:.2f}\033[0m kg CO\u2082")
    print(f"Suggestion-based Model Carbon Footprint: {future_prediction:.2f} kg CO\u2082")
    print(f"\033[32m--- Estimated daily reduction of {reduction:.2f} kg CO\u2082 ({percent:.2f}%) ---\033[0m")

    print("\nBehavior Summary")
    print(f"Distance: {user['distance_km']:.1f} km "
      f"(Average: {behaviors['distance_km'].mean():.1f})")
    print(f"Electricity: {user['electricity_kwh']:.1f} kWh "
      f"(Average: {behaviors['electricity_kwh'].mean():.1f})")
    print(f"Renewable usage: {user['renewable_usage_pct']}%")
    print(f"Screen time: {user['screen_time_hours']} hours")
    print(f"Waste generated: {user['waste_generated_kg']} kg "
      f"(Average: {behaviors['waste_generated_kg'].mean():.1f})")

    print(f"Eco actions: {user['eco_actions']}")

    print("\nWhy this profile?")

    if user["distance_km"] > behaviors["distance_km"].mean():
        print("• Travels farther than the average user.")

    if user["electricity_kwh"] > behaviors["electricity_kwh"].mean():
        print("• Uses more electricity than average.")

    if user["eco_actions"] < behaviors["eco_actions"].mean():
        print("• Performs fewer eco-friendly actions than average.")

    print(f"\nUser Recommendations")
    user_rec = recommend_user(user)
    for rec in user_rec:
        print(f"• {rec}")
    else:
        print("• No additional personalized recommendations.")

    print("\nCluster-Based Recommendedations")
    for rec in recommend(user["cluster"]):
        print(f"• {rec}")

    print()

User 665
Sustainability Profile: Lesser Consumers
Current Prediction Carbon Footprint: 5.85 kg CO₂
Suggestion-based Model Carbon Footprint: 5.77 kg CO₂
--- Estimated daily reduction of 0.08 kg CO₂ (1.29%) ---

Behavior Summary
Distance: 9.6 km (Average: 9.1)
Electricity: 4.0 kWh (Average: 6.0)
Renewable usage: 0%
Screen time: 5.4 hours
Waste generated: 0.56 kg (Average: 0.7)
Eco actions: 1

Why this profile?
• Travels farther than the average user.
• Performs fewer eco-friendly actions than average.

User Recommendations
• Use public transportation multiple times per week.
• No additional personalized recommendations.

Cluster-Based Recommendedations
• Increase biking or walking distances
• Increase daily eco-friendly actions
• Reduce screen time to save electricity

User 624
Sustainability Profile: Large Consumers
Current Prediction Carbon Footprint: 9.52 kg CO₂
Suggestion-based Model Carbon Footprint: 8.96 kg CO₂
--- Estimated daily reduction of 0.56 kg CO₂ (5.90%) ---

Behavior Summ

Conclusions

This recommendation framework demonstrates how predictive analytics can support sustainability inititives by turning user data into personalized recommendations. This model can be altered to fit many unique datasets by simply switching definitions and recommendations around.

While the recommendations are built on estimations rather than observed changed, this project shows how machine learning can be combined with real data sets to provide beneficial sustainable impact.

Future improvements could include a more personalized algorith and an expanded intake of personal habits, able to understand a wider variety of personal situations and habits. Utilizing an app or website interphase, the technology could be added to existing sites to add interactive capabilities. 